In [ ]:
# Install gdown to download from Google Drive
!pip install -q gdown

# Import libraries
import gdown
import tensorflow as tf

# Download the model file from Google Drive
file_id = '1oork83eYanFkW8pxj6EuTy76niesn3FN'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'EfficientNetB1.keras', quiet=False)

# Load the downloaded model
model = tf.keras.models.load_model('EfficientNetB1.keras')

# Now the model is loaded and ready to use!

Downloading...
From (original): https://drive.google.com/uc?id=1oork83eYanFkW8pxj6EuTy76niesn3FN
From (redirected): https://drive.google.com/uc?id=1oork83eYanFkW8pxj6EuTy76niesn3FN&confirm=t&uuid=f1810e50-84da-48a1-9011-8f806e9510b2
To: /content/EfficientNetB1.keras
100%|██████████| 94.1M/94.1M [00:00<00:00, 154MB/s]


In [ ]:
import tensorflow as tf


def mi_fgsm(model, x, y, epsilon=8.0, T=10, mu=1.0):
    """
    MI-FGSM attack for models that take unnormalized inputs in [0, 255].

    Args:
        model: A tf.keras.Model that outputs logits.
        x: Input image tensor (float32) in [0, 255], shape (batch_size, H, W, C).
        y: Ground-truth label (int scalar, int vector, or one-hot),
           with shape (), (batch_size,), (num_classes,), or (batch_size, num_classes).
        epsilon: Perturbation bound (L∞ norm), e.g. 8.0 for images in [0, 255].
        T: Number of iterations.
        mu: Momentum decay factor.

    Returns:
        x_star: Adversarial image tensor in [0, 255], same shape as x.
    """
    # Scale epsilon to pixel range
    epsilon = epsilon * 255.0

    # Cast inputs
    x = tf.cast(x, tf.float32)
    x_star = tf.identity(x)
    batch_size = tf.shape(x)[0]

    # Step size per iteration
    alpha = epsilon / float(T)

    # Initialize momentum buffer
    g = tf.zeros_like(x)

    # Prepare labels
    num_classes = model.output_shape[-1]
    y = tf.cast(y, tf.int32)
    y_shape = y.shape
    # Case 1: scalar label
    if y_shape.ndims == 0:
        y = tf.expand_dims(y, 0)
        y = tf.one_hot(y, depth=num_classes)
    # Case 2: vector
    elif y_shape.ndims == 1:
        # If length equals num_classes, assume one-hot vector
        if y_shape[0] == num_classes:
            y = tf.expand_dims(tf.cast(y, tf.float32), 0)
        else:
            # integer class indices
            y = tf.one_hot(y, depth=num_classes)
    # Case 3: already (batch_size, num_classes)
    elif y_shape.ndims == 2 and y_shape[1] == num_classes:
        y = tf.cast(y, tf.float32)
    else:
        raise ValueError(f"Unsupported label shape: {y_shape}")

    loss_object = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

    # Iterative attack
    for _ in range(T):
        with tf.GradientTape() as tape:
            tape.watch(x_star)
            logits = model(x_star)
            loss = loss_object(y, logits)

        # Compute gradient
        grad = tape.gradient(loss, x_star)

        # Normalize by L1 norm
        grad_norm = tf.reduce_sum(
            tf.abs(grad),
            axis=list(range(1, len(grad.shape))),
            keepdims=True
        )
        grad_norm = tf.maximum(grad_norm, 1e-8)
        normalized_grad = grad / grad_norm

        # Momentum update
        g = mu * g + normalized_grad

        # Perturbation step
        x_star = x_star + alpha * tf.sign(g)

        # Project back into epsilon-ball
        x_star = tf.clip_by_value(x_star, x - epsilon, x + epsilon)

        # Clip to valid pixel range
        x_star = tf.clip_by_value(x_star, 0.0, 255.0)

    return x_star





In [ ]:
import zipfile
import os

zip_path = "/content/Attack_testdata.zip"  # Replace with your ZIP filename
extract_path = "extracted_data"  # You can name this anything

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [ ]:
image_extensions = ('.jpg', '.jpeg', '.png')

image_count = 0
for root, dirs, files in os.walk(extract_path):
    image_count += len([f for f in files if f.lower().endswith(image_extensions)])

print(f"Total number of image samples: {image_count}")


Total number of image samples: 516


In [ ]:
import pandas as pd
from GTSRB_utils import GTSRB_CLASSES
# Load the CSV file
labels_df = pd.read_csv("/content/extracted_data/Attack_Test.csv")

# Display the first few rows to understand the structure
print(labels_df.head())

# Assuming the CSV has columns like 'filename' and 'label'
# Adjust column names based on the actual CSV structure
image_filenames = labels_df['Path'].values  # e.g., '12617.png'
labels = labels_df['ClassId'].values  # e.g., 0 or 1, or class names


import tensorflow as tf
import numpy as np

# Define image parameters
img_height = 240  # Adjust to match your model's expected input size
img_width = 240
batch_size = 32

# Function to load and preprocess images (using tf operations)
def load_and_preprocess_image(image_path):
    # Read the image file
    img = tf.io.read_file(image_path)  # image_path is already a tensor string
    img = tf.image.decode_png(img, channels=3)  # Assuming RGB images

    # Resize to target size
    img = tf.image.resize(img, [img_height, img_width])

    return img

# Create a dataset from the image filenames and labels
def create_dataset(image_filenames, labels):
    # Convert filenames and labels to tensors
    base_dir = "/content/extracted_data"
    image_paths = [os.path.join(base_dir, fname) for fname in image_filenames]
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    # Map the load_and_preprocess_image function
    dataset = dataset.map(
        lambda path, label: (load_and_preprocess_image(path), label),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Shuffle, batch, and prefetch
    dataset = dataset.shuffle(buffer_size=len(image_filenames))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Create the dataset
dataset = create_dataset(image_filenames, labels)

   Width  Height  Roi.X1  Roi.Y1  Roi.X2  Roi.Y2  ClassId       Path
0     39      39       6       5      34      34        0  00243.png
1     39      40       5       5      34      35        0  00778.png
2     35      36       5       6      30      31        0  04726.png
3     49      51       6       5      44      46        0  06854.png
4     34      34       6       6      29      29        0  02045.png


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = []
y_pred = []

for batch_images, batch_labels in dataset:
    preds = model.predict(batch_images)
    pred_classes = tf.argmax(preds, axis=1).numpy()

    y_true.extend(batch_labels.numpy())
    y_pred.extend(pred_classes)

# Evaluate
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=GTSRB_CLASSES.values() if isinstance(GTSRB_CLASSES, dict) else class_mapping))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Accuracy: 0.9632

Classification Report:
                                              precision    recall  f1-score   support

                              Speed limit 20       1.00      1.00      1.00        12
                              Speed limit 30       1.00      1.00      1.00        12
                              Speed limit 50  

In [ ]:
import os
import tensorflow as tf
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from GTSRB_utils import GTSRB_CLASSES

# Configuration
new_directory_path = "/content/extracted_data"
labels_csv = os.path.join(new_directory_path, "Attack_Test.csv")
model_path = "/content/EfficientNetB1.keras"
output_root = "/content/MI_FGSM_Test_adv_examples"

# Hyperparameter grids
epsilons = [0.01]  # Example epsilon values
mus = [0.5, 1.0]             # Example momentum factors
T = 10                             # Number of iterations for MI-FGSM

# Image preprocessing parameters
img_height = 240
img_width = 240
batch_size = 32

# Create output root if it doesn't exist
os.makedirs(output_root, exist_ok=True)

# Load labels dataframe
df = pd.read_csv(labels_csv)

# Load model
model = tf.keras.models.load_model(model_path)

# List all images
image_filenames = [f for f in os.listdir(new_directory_path) if f.lower().endswith('.png')]

# Function to save image tensor
def save_image_tensor(img_tensor, path):
    img_uint8 = np.clip(img_tensor, 0, 255).astype(np.uint8)
    tf.keras.preprocessing.image.save_img(path, img_uint8)

# Loop over hyperparameter variants
for eps in epsilons:
    for mu in mus:
        variant_name = f"eps_{eps}_mu_{mu}"
        variant_dir = os.path.join(output_root, variant_name)
        adv_dir = os.path.join(variant_dir, "images")
        os.makedirs(adv_dir, exist_ok=True)

        records = []  # to store csv rows

        # Process each image
        for img_name in tqdm(image_filenames, desc=variant_name):
            row = df[df['Path'] == img_name]
            if row.empty:
                continue
            true_label = int(row.iloc[0]['ClassId'])

            # Load and preprocess image
            img_path = os.path.join(new_directory_path, img_name)
            img_raw = tf.io.read_file(img_path)
            img = tf.image.decode_png(img_raw, channels=3)
            img = tf.image.resize(img, [img_height, img_width])
            img = tf.cast(img, tf.float32)
            img_batch = tf.expand_dims(img, axis=0)

            # Generate adversarial example
            adv_batch = mi_fgsm(model, img_batch, tf.convert_to_tensor([true_label]),
                                epsilon=eps, T=T, mu=mu)
            adv_img = adv_batch[0].numpy()

            # Predict on adversarial
            adv_logits = model.predict(adv_batch)
            adv_pred = int(tf.argmax(adv_logits, axis=1).numpy()[0])

            # Save adversarial image
            save_path = os.path.join(adv_dir, img_name)
            save_image_tensor(adv_img, save_path)

            # Record
            records.append({'Image': img_name,
                            'TrueLabel': true_label,
                            'PredAdv': adv_pred})

        # Convert records to DataFrame and save
        df_records = pd.DataFrame(records)
        csv_path = os.path.join(variant_dir, "predictions.csv")
        df_records.to_csv(csv_path, index=False)

        # Compute and report metrics
        y_true_adv = df_records['TrueLabel'].values
        y_pred_adv = df_records['PredAdv'].values

        acc_adv = accuracy_score(y_true_adv, y_pred_adv)
        print(f"\nAdversarial Test Accuracy (MI-FGSM, {variant_name}): {acc_adv:.4f}")

        print("\nClassification Report (MI-FGSM):")
        print(classification_report(y_true_adv, y_pred_adv))

        print("\nConfusion Matrix (MI-FGSM):")
        print(confusion_matrix(y_true_adv, y_pred_adv))

eps_0.01_mu_0.5:   0%|          | 0/516 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:666: UserWarning: "`categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step


eps_0.01_mu_0.5:   0%|          | 1/516 [00:19<2:43:48, 19.08s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step


eps_0.01_mu_0.5:   0%|          | 2/516 [00:27<1:52:11, 13.10s/it]

...

eps_0.01_mu_1.0: 100%|█████████▉| 515/516 [1:12:27<00:08,  8.46s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


eps_0.01_mu_1.0: 100%|██████████| 516/516 [1:12:35<00:00,  8.44s/it]


Adversarial Test Accuracy (MI-FGSM, eps_0.01_mu_1.0): 0.2403

Classification Report (MI-FGSM):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        12
           1       0.64      0.75      0.69        12
           2       0.11      0.08      0.10        12
           3       0.00      0.00      0.00        12
           4       0.07      0.17      0.10        12
           5       0.00      0.00      0.00        12
           6       0.17      0.08      0.11        12
           7       0.03      0.08      0.05        12
           8       0.08      0.08      0.08        12
           9       0.21      0.33      0.26        12
          10       0.00      0.00      0.00        12
          11       0.15      0.17      0.16        12
          12       0.50      0.17      0.25        12
          13       0.75      0.75      0.75        12
          14       0.60      0.25      0.35        12
          15       0.73      0.67      


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Zip up the adv_variants folder
!zip -r /content/MI_FGSM_Test_adv_examples.zip /content/MI_FGSM_Test_adv_examples

# 3) Copy the zip to your Drive (e.g. into MyDrive root)
!cp /content/MI_FGSM_Test_adv_examples.zip /content/drive/MyDrive/



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  adding: content/MI_FGSM_Test_adv_examples/ (stored 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/ (stored 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/ (stored 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/08679.png (deflated 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/04479.png (deflated 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/09834.png (deflated 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/09390.png (deflated 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/05077.png (deflated 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/10518.png (deflated 0%)
  adding: content/MI_FGSM_Test_adv_examples/eps_0.01_mu_0.5/images/01613.png (deflated 0%)
  adding: content/MI_FGS